# 05 — Injection d'un foyer de démo (moment fort)

Injecte un foyer chaud **déterministe** près d'une commune dense (Bormes-les-Mimosas) pour déclencher : priorité élevée → **Activator** → alerte **Teams** → mise à jour de la carte. Trois options de livraison ; l'**Option 1 (KQL inline)** est la plus fiable pour la scène. Données de démo synthétiques.

In [ ]:
from datetime import datetime, timezone
DEMO_FIRE = {
    "detection_id": "DEMO-BORMES-001",
    "latitude": 43.150, "longitude": 6.340,     # proche de Bormes-les-Mimosas (commune dense)
    "frp": 92.5,                                  # puissance radiative élevée = foyer intense
    "confidence": "h",
    "acq_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    "acq_time": datetime.now(timezone.utc).strftime("%H%M"),
    "daynight": "D",
    "source": "DEMO_INJECTION",
    "ingest_ts": datetime.now(timezone.utc).isoformat(),
}
print(DEMO_FIRE)

## Option 1 — KQL inline (recommandé, zéro dépendance)
Copie la commande imprimée et exécute-la dans un **KQL Queryset** de `EH_Wildfire`. Instantané et déterministe.

In [ ]:
cols = ["detection_id","latitude","longitude","frp","confidence","acq_date","acq_time","daynight","source","ingest_ts"]
row = ",".join(str(DEMO_FIRE[c]) for c in cols)
print(".ingest inline into table FireDetections <|\n" + row)

## Option 2 — Écriture Lakehouse (batch, toujours disponible)
Attache `LH_Wildfire` au notebook.

In [ ]:
import pandas as pd
sdf = spark.createDataFrame(pd.DataFrame([DEMO_FIRE]))
sdf.write.format("delta").mode("append").saveAsTable("bronze_fire_detections")
print("Foyer de démo ajouté à bronze_fire_detections.")

## Option 3 — Temps réel via Eventstream (optionnel)
Récupère la chaîne de connexion **Event Hub** du *Custom endpoint* de `ES_Wildfire`. Requiert `azure-eventhub` (repli : Option 1).

In [ ]:
EVENTHUB_CONN_STR = "REMPLACER_PAR_CONNECTION_STRING_CUSTOM_ENDPOINT"
EVENTHUB_NAME     = "REMPLACER_PAR_ENTITY_PATH"
try:
    import json
    from azure.eventhub import EventHubProducerClient, EventData
    producer = EventHubProducerClient.from_connection_string(EVENTHUB_CONN_STR, eventhub_name=EVENTHUB_NAME)
    with producer:
        batch = producer.create_batch()
        batch.add(EventData(json.dumps(DEMO_FIRE)))
        producer.send_batch(batch)
    print("Événement poussé vers l'Eventstream.")
except Exception as e:
    print(f"Push Eventstream indisponible ({e}). Utilise l'Option 1 (KQL inline) comme repli.")